<a href="https://colab.research.google.com/github/espickle1/boltz-2/blob/main/src/boltz_predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 1. Install Boltz-2 (GPU)
#@markdown If prompted, click **RESTART RUNTIME** and then continue to the next cell.
%%capture
!pip install boltz[cuda] -U

In [ ]:
#@title 2. Import dependencies
!pip -q install ipywidgets
from google.colab import output, files
output.enable_custom_widget_manager()

import ipywidgets as W
from IPython.display import display, clear_output
import os, shlex, subprocess, sys, shutil, pathlib, time

In [ ]:
#@title 3. Upload YAML inputs & configure parameters
print("Select and upload prediction YAML file(s):")
uploaded = files.upload()

out_dir_txt = W.Text(
    value="/content/boltz_results",
    description="Output path:",
    layout=W.Layout(width="320px"),
)

recycling_steps = W.IntText(value=8, description="Recycling steps:", layout=W.Layout(width="220px"))
sampling_steps = W.IntText(value=64, description="Sampling steps:", layout=W.Layout(width="220px"))
diffusion_samples = W.IntText(value=5, description="Diffusion samples:", layout=W.Layout(width="220px"))
step_scale = W.FloatText(value=1.7, description="Step scale:", layout=W.Layout(width="220px"))
sampling_steps_affinity = W.IntText(value=64, description="Sampling (affinity):", layout=W.Layout(width="220px"))
diffusion_samples_affinity = W.IntText(value=4, description="Diffusion (affinity):", layout=W.Layout(width="220px"))
use_msa_server = W.Checkbox(value=True, description="Use MSA server", layout=W.Layout(width="220px"))

display(W.VBox([
    out_dir_txt,
    W.HBox([recycling_steps, sampling_steps, diffusion_samples]),
    W.HBox([step_scale, sampling_steps_affinity, diffusion_samples_affinity]),
    use_msa_server,
]))

In [ ]:
#@title 4. Run Boltz-2 batch prediction
def stream_cmd(cmd_list):
    """Run a command and stream stdout+stderr line-by-line. Returns exit code."""
    proc = subprocess.Popen(
        cmd_list,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )
    for line in proc.stdout:
        print(line, end="")
        sys.stdout.flush()
    return proc.wait()

def zip_and_download(run_dir: pathlib.Path):
    """Zip the run_dir and trigger a browser download."""
    run_dir = run_dir.resolve()
    zip_path = shutil.make_archive(str(run_dir), "zip", root_dir=str(run_dir))
    print(f"\nZipped results -> {zip_path}")
    files.download(zip_path)

base_out_dir = pathlib.Path((out_dir_txt.value or "/content/boltz_results").strip())
base_out_dir.mkdir(parents=True, exist_ok=True)

yaml_files = sorted(uploaded.keys())
print(f"Found {len(yaml_files)} YAML file(s)")

succeeded, failed = [], []

for i, fname in enumerate(yaml_files, 1):
    yaml_path = pathlib.Path(fname).resolve()
    run_dir = base_out_dir / yaml_path.stem
    run_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"[{i}/{len(yaml_files)}] {yaml_path.name}")
    print(f"{'='*60}")

    cmd = [
        "boltz", "predict",
        str(yaml_path),
        "--out_dir", str(run_dir),
        "--recycling_steps", str(recycling_steps.value),
        "--sampling_steps", str(sampling_steps.value),
        "--diffusion_samples", str(diffusion_samples.value),
        "--step_scale", str(step_scale.value),
        "--sampling_steps_affinity", str(sampling_steps_affinity.value),
        "--diffusion_samples_affinity", str(diffusion_samples_affinity.value),
    ]
    if use_msa_server.value:
        cmd.append("--use_msa_server")

    print("$", " ".join(shlex.quote(x) for x in cmd))

    rc = stream_cmd(cmd)
    print(f"\n[boltz exited with code {rc}]")

    if rc != 0:
        print(f"FAILED: {yaml_path.name}")
        failed.append(yaml_path.name)
        continue

    succeeded.append(yaml_path.name)
    time.sleep(0.5)
    try:
        zip_and_download(run_dir)
    except Exception as e:
        print(f"Could not zip/download results: {e}")

print(f"\n{'='*60}")
print(f"DONE: {len(succeeded)} succeeded, {len(failed)} failed out of {len(yaml_files)}")
if failed:
    print(f"Failed: {', '.join(failed)}")